# Population by State and Age

Source: Destatis, GENESIS-Online, table [**12411-0014**](https://genesis.destatis.de/datenbank/online?sequenz=tabelleErgebnis&selectionname=12411-0014): population by federal state, nationality, sex and single year of age.

First step: load the raw file and look at it.

In [10]:
import pandas as pd

population_raw = pd.read_csv(
    "data/population_states_age.csv",
    sep=";",               # GENESIS uses a semicolon as separator (German CSV format)
    encoding="latin-1",    # German characters (ä, ö, ü, ß) are stored in Latin-1;
                           # if they look broken, try "utf-8-sig" instead
    header=None,           # the first lines are titles, not column names,
                           # so we read everything as plain rows (columns get numbers 0, 1, 2, ...)
    on_bad_lines="skip",   # title and footnote lines have fewer columns than the data rows;
                           # skip them instead of stopping with an error
)

print(population_raw.shape)   # (number of rows, number of columns)
population_raw.head(20)       # show the first 20 rows to find where the real header starts

(5, 1)


,0
0,ï»¿Tabelle: 12411-0014
1,__________
2,From 2011: Results based on the 2011 Census.\n...
3,"Â© Federal Statistical Office, Wiesbaden 2026"
4,created: 2026-10-20 / 12:20:38


In [11]:
# Read the file as plain text lines (no pandas) to see its real structure
with open("data/population_states_age.csv", encoding="utf-8-sig") as population_file:
    population_lines = population_file.read().splitlines()

print("total lines:", len(population_lines))

# Show the first 15 lines with their line number
for line_number, line in enumerate(population_lines[:15]):
    print(line_number, line[:150])   # cut long lines at 150 characters

total lines: 1483
0 Tabelle: 12411-0014
1 Population: Länder, reference date, nationality, sex, age;;;;;;;;;;;
2 Current updating of population figures;;;;;;;;;;;;;;;;;;;;
3 Population (number);;;;;;;;;;;;;;;;;;;;
4 ;;;Germans;Germans;Germans;Germans;Germans;Germans;Foreigners;Foreigners;Foreigners;Foreigners;Foreigners;Foreigners;Total;Total;Total;Total;Total;Tot
5 ;;;Male;Male;Female;Female;Total;Total;Male;Male;Female;Female;Total;Total;Male;Male;Female;Female;Total;Total
6 2025-12-31;Baden-Württemberg;under 1 year;42030;e;40107;e;82137;e;5948;e;5462;e;11410;e;47978;e;45569;e;93547;e
7 2025-12-31;Baden-Württemberg;1 year;43341;e;40974;e;84315;e;7117;e;6639;e;13756;e;50458;e;47613;e;98071;e
8 2025-12-31;Baden-Württemberg;2 years;42927;e;40835;e;83762;e;8289;e;7880;e;16169;e;51216;e;48715;e;99931;e
9 2025-12-31;Baden-Württemberg;3 years;45868;e;43429;e;89297;e;8700;e;8281;e;16981;e;54568;e;51710;e;106278;e
10 2025-12-31;Baden-Württemberg;4 years;49943;e;48701;e;98644;e;8972;e;8539;e;1

In [12]:
import pandas as pd

# Own column names: date, state, age, then 6 groups, each followed by its quality flag
population_columns = [
    "reference_date", "state", "age",
    "german_male", "flag_1", "german_female", "flag_2", "german_total", "flag_3",
    "foreign_male", "flag_4", "foreign_female", "flag_5", "foreign_total", "flag_6",
    "total_male", "flag_7", "total_female", "flag_8", "total", "flag_9",
]

population_read = pd.read_csv(
    "data/population_states_age.csv",
    sep=";",
    encoding="utf-8-sig",      # the file is UTF-8 (this also removes the strange "ï»¿" at the start)
    skiprows=6,                # skip 4 title lines + 2 header lines
    header=None,               # no header row left in the file
    names=population_columns,  # use our own column names instead
)

# Keep only real data rows (they start with a date like 2025-12-31) and drop the flag columns
is_data_row = population_read["reference_date"].str.match(r"\d{4}-\d{2}-\d{2}", na=False)
value_columns = [c for c in population_columns if not c.startswith("flag")]
population_states_age = population_read.loc[is_data_row, value_columns]

print(population_states_age.shape)
print("states:", population_states_age["state"].nunique())
print("age values:", population_states_age["age"].unique()[:3], "...", population_states_age["age"].unique()[-3:])
population_states_age.head()

(1472, 12)
states: 16
age values: <StringArray>
['under 1 year', '1 year', '2 years']
Length: 3, dtype: str ... <StringArray>
['89 years', '90 years and over', 'Total']
Length: 3, dtype: str


,reference_date,state,age,german_male,german_female,german_total,foreign_male,foreign_female,foreign_total,total_male,total_female,total
0,2025-12-31,Baden-Württemberg,under 1 year,42030.0,40107.0,82137.0,5948.0,5462.0,11410.0,47978.0,45569.0,93547.0
1,2025-12-31,Baden-Württemberg,1 year,43341.0,40974.0,84315.0,7117.0,6639.0,13756.0,50458.0,47613.0,98071.0
2,2025-12-31,Baden-Württemberg,2 years,42927.0,40835.0,83762.0,8289.0,7880.0,16169.0,51216.0,48715.0,99931.0
3,2025-12-31,Baden-Württemberg,3 years,45868.0,43429.0,89297.0,8700.0,8281.0,16981.0,54568.0,51710.0,106278.0
4,2025-12-31,Baden-Württemberg,4 years,49943.0,48701.0,98644.0,8972.0,8539.0,17511.0,58915.0,57240.0,116155.0


In [13]:
# Check 1: total = Germans + foreigners, and total = male + female (in every row)
check_nationality = population_states_age["total"] == population_states_age["german_total"] + population_states_age["foreign_total"]
check_sex = population_states_age["total"] == population_states_age["total_male"] + population_states_age["total_female"]
print("rows where total != Germans + foreigners:", (~check_nationality).sum())
print("rows where total != male + female:      ", (~check_sex).sum())

# Check 2: the "Total" age row of each state = sum of all single ages of that state
is_total_row = population_states_age["age"] == "Total"
sum_of_ages = population_states_age.loc[~is_total_row].groupby("state")["total"].sum()
total_row = population_states_age.loc[is_total_row].set_index("state")["total"]
total_comparison = pd.DataFrame({"sum_of_ages": sum_of_ages, "total_row": total_row})
total_comparison["difference"] = total_comparison["sum_of_ages"] - total_comparison["total_row"]
total_comparison

rows where total != Germans + foreigners: 0
rows where total != male + female:       0


,sum_of_ages,total_row,difference
state,,,
Baden-Württemberg,11238761.0,11238761.0,0.0
Bayern,13245503.0,13245503.0,0.0
Berlin,3700577.0,3700577.0,0.0
Brandenburg,2551889.0,2551889.0,0.0
Bremen,706915.0,706915.0,0.0
Hamburg,1869473.0,1869473.0,0.0
Hessen,6278594.0,6278594.0,0.0
Mecklenburg-Vorpommern,1573685.0,1573685.0,0.0
Niedersachsen,8000826.0,8000826.0,0.0


In [14]:
# State numbers 1-16, same codes as ULAND in the Unfallatlas
STATE_TO_ULAND = {
    "Schleswig-Holstein": 1, "Hamburg": 2, "Niedersachsen": 3, "Bremen": 4,
    "Nordrhein-Westfalen": 5, "Hessen": 6, "Rheinland-Pfalz": 7, "Baden-Württemberg": 8,
    "Bayern": 9, "Saarland": 10, "Berlin": 11, "Brandenburg": 12,
    "Mecklenburg-Vorpommern": 13, "Sachsen": 14, "Sachsen-Anhalt": 15, "Thüringen": 16,
}

# Keep only the age rows (drop "Total") and only the columns we need
population_simple = population_states_age.loc[
    population_states_age["age"] != "Total", ["state", "age", "total"]
].copy()

population_simple["ULAND"] = population_simple["state"].map(STATE_TO_ULAND)   # state number 1-16
population_simple["total"] = population_simple["total"].astype(int)            # whole numbers
population_simple = population_simple[["ULAND", "state", "age", "total"]]      # column order

print(population_simple.shape)                                         # expected: (1456, 4) = 16 states x 91 ages
print("states without number:", population_simple["ULAND"].isna().sum())   # expected: 0
population_simple.head()

(1456, 4)
states without number: 0


,ULAND,state,age,total
0,8,Baden-Württemberg,under 1 year,93547
1,8,Baden-Württemberg,1 year,98071
2,8,Baden-Württemberg,2 years,99931
3,8,Baden-Württemberg,3 years,106278
4,8,Baden-Württemberg,4 years,116155


In [15]:
# Each state with its number, taken from population_simple
population_simple[["ULAND", "state"]].drop_duplicates().sort_values("ULAND").reset_index(drop=True)

,ULAND,state
0,1,Schleswig-Holstein
1,2,Hamburg
2,3,Niedersachsen
3,4,Bremen
4,5,Nordrhein-Westfalen
5,6,Hessen
6,7,Rheinland-Pfalz
7,8,Baden-Württemberg
8,9,Bayern
9,10,Saarland


In [16]:
population_simple.head()

,ULAND,state,age,total
0,8,Baden-Württemberg,under 1 year,93547
1,8,Baden-Württemberg,1 year,98071
2,8,Baden-Württemberg,2 years,99931
3,8,Baden-Württemberg,3 years,106278
4,8,Baden-Württemberg,4 years,116155


In [17]:
pip install geopandas

/Users/zina/Documents/GitHub/The-Random-Forest-Rangers/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. Total population per state (sum of all ages from population_simple)
population_per_state = population_simple.groupby(["ULAND", "state"], as_index=False)["total"].sum()

# 2. Borders of the 16 German states (GeoJSON from GitHub, state names in German like in our table)
GERMANY_STATES_URL = "https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/main/2_bundeslaender/4_niedrig.geo.json"
germany_states_map = gpd.read_file(GERMANY_STATES_URL)

# 3. Join the population to the map by state name
population_map = germany_states_map.merge(population_per_state, left_on="name", right_on="state", how="left")
print("states without population:", population_map["total"].isna().sum())   # expected: 0

# 4. Draw the map: darker blue = more people
fig, ax = plt.subplots(figsize=(9, 11))
population_map.plot(
    column="total", cmap="Blues", edgecolor="white", linewidth=0.8, legend=True, ax=ax,
    legend_kwds={"label": "Population (millions)", "shrink": 0.6,
                 "format": FuncFormatter(lambda value, _: f"{value / 1e6:.0f}")},
)

# 5. Label every state with its ULAND number and population;
#    the three small city states get their label outside with a thin line
label_offsets = {"Berlin": (1.0, 0.45), "Bremen": (-1.6, 0.6), "Hamburg": (0.9, 0.35)}
for _, row in population_map.iterrows():
    point = row.geometry.representative_point()
    label = f"{row['ULAND']}\n{row['total'] / 1e6:.1f} M"
    if row["state"] in label_offsets:
        dx, dy = label_offsets[row["state"]]
        ax.annotate(label, xy=(point.x, point.y), xytext=(point.x + dx, point.y + dy), fontsize=8,
                    arrowprops=dict(arrowstyle="-", color="#555555", lw=0.6))
    else:
        text_color = "white" if row["total"] > 10e6 else "black"   # white text on dark states
        ax.text(point.x, point.y, label, fontsize=8, ha="center", va="center", color=text_color)

ax.set_title("Population by federal state, 31.12.2025\n(number = ULAND code)", fontsize=13)
ax.set_axis_off()
ax.set_aspect(1.5)   # keeps Germany's shape from looking squashed
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'geopandas'